




??????
use “synthetic patient records” throughout the README rather than “patient data,”


# The Knowledge OR DATASET Base for Clinical Synopsis: Source-Grounded Patient Summaries

We are interested in the **oncology** setting. We're using **synthetic patient data** from generated by Synthea, specifically, the **mCODE test data** built from Synthea because it includes both cancer-related and cancer-unrelated **broader record of a patient's history**, including non-cancer **encounters** (i.e., any professional interactions between a patient and a healthcare provider), **conditions**, and **medications**.

The FHIR format for **Electronic Health Records (EHRs)** is the standard way to represent structured clinical data, which makes the synthetic oncology records in this project realistic and easier to process and reuse across systems. 

**mCODE** (Minimal Common Oncology Data Elements) is an open-source, HL7 FHIR-based data standard for cancer patient data, developed by the American Society of Clinical Oncology (ASCO) and the MITRE Corporation.

We can obtain standardized, synthetic patient data from the HL7 Confluence repository for mCODE Test Data. "Because of the way that Synthea outputs FHIR records, it is not possible at this time to output mCODE patients directly out of Synthea. So these patients have been post-processed using the fhir-mapper."
https://confluence.hl7.org/spaces/COD/pages/80119851/mCODE+Test+Data
(Not yet available are pathologic staging, genetics/genomics, metastasis records.)


For this project we selected 50 patients from the **STU2 breast cancer, lifetime/longitudinal EERs dataset** using a fixed, step-by-step rule based on medical complexity (how often a patient was seen, i.e., "encounter count", and how long they were followed, i.e., "follow-up time"). See `data/processed/mcode_breast_sample_50_manifest.csv` for details. (The breast cancer dataset from mCode is more densely populated than the mixed-cancer set, because it provides broad patient history.)

Our challenge is not only retrieval, but a source-grounded synthesis across mixed document types. (Which makes our evalution focus on patient-summary quality and source-grounding.) Therefore, for each patient we want mixed documents such as encounter summaries, lab results, medication lists, and oncology-related reports, so that the RAG application retrieves from both text and structured metadata.


Markdown + CSV + Excel (later)


We turn those records into turn into a patient corpus with a small metadata database.



For your current RAG design, you’re rightly treating `patient_overview.md` as the **trusted summary source** and then offering CSVs as “click for more detail,” rather than trying to have the LLM synthesize everything from raw tables every time.

- “Recent” = **most recent N entries by date**, where:
  - N = 10 for conditions, meds, procedures, reports.
  - N = 12 for observations.
  - N = 8 for encounters.
- There is **no explicit time window** (like “last year”); it is strictly a **top-N cutoff based on the available dates**.
- That’s why older long-term history in the CSVs may be missing from `patient_overview.md`: it only shows the latest slice per category.




## Select 50 patients

The script `clinical_synopsis/scripts/sample_mcode_patients.py` uses the raw mCODE breast cancer patient json bundles in `data/raw/longitudinalMCODEBreast` to compute per‑patient statistics (counts of encounters, observations, conditions, follow‑up duration, a “complexity” score), assigns each patient to a **low/medium/high complexity** bucket, and then draws a stratified random sample of 50 patients. The full stats table and the 50‑patient manifest are saved as CSV files in `data/processed`.

```bash
uv run clinical_synopsis/scripts/sample_mcode_patients.py
```
Output:
```
Found 258 JSON files total
Kept 256 patient bundles
Skipped 2 non-patient bundles

Wrote patient stats to: data/processed/mcode_breast_patient_stats.csv
Wrote 50-patient manifest to: data/processed/mcode_breast_sample_50_manifest.csv

Sample bucket counts:
complexity_bucket
low       17
high      17
medium    16
```

`create_prototype_buckets.py` reads the manifest `data/processed/mcode_breast_sample_50_manifest.csv`, copies the selected 50 JSON files into `data/prototype/sample50`, and also creates low, medium, and high folders.

```bash
uv run clinical_synopsis/scripts/create_prototype_buckets.py
```
Output:
```
Copied 50 files.

Bucket counts in manifest:
complexity_bucket
low       17
high      17
medium    16

Output folders:
- data/prototype/sample50
- data/prototype/low
- data/prototype/medium
- data/prototype/high
```

In [12]:
!find ../data/prototype/high -type f | wc -l

17


What comes next
So I’d recommend this order:
**Canonical extraction layer** Parse each FHIR bundle into structured patient-level tables or JSON objects: demographics, encounters, conditions, medications, observations, procedures, reports.[landing]
**Derived document layer** From those normalized records, generate the documents your RAG app will actually search over, such as:
patient_summary.md
encounters.md
medications.xlsx
labs.xlsx
oncology_timeline.md
**Chunking + provenance metadata**
Chunk those derived documents and attach patient ID, filename, document type, dates, and source pointers. Provenance is a key part of reliable RAG pipelines.[landing]

Think of it as:
raw bundle = source of truth,
canonical extraction = clean internal representation,
derived documents = retrieval-friendly views for Clinical Synopsis.



## Extract tabular data for each patient

`extract_mcode_patient_tables.py` extracts each sampled FHIR bundle into a per-patient folder of CSV and JSON files. For each patient in `data/prototype/sample50` it creates:
```
data/interim/sample50/<patient_id>/patient.csv
.../encounters.csv
.../conditions.csv
.../observations.csv
.../medication_requests.csv
.../medication_administrations.csv
.../procedures.csv
.../diagnostic_reports.csv
.../bundle_metadata.json
```



```bash
uv run clinical_synopsis/scripts/extract_mcode_patient_tables.py
```
Output:
```
Processed 50 patient files into data/interim/sample50
```

## Derive mock EHRs

`generate_derived_documents.py` creates the documents that the RAG searches over, i.e., the mock health records imitating the structure and content of real clinical records.

It uses the following logic for each patient:
- Always generate:
    - patient_overview.md
    - encounters.csv
    - conditions.csv
    - observations.csv
- Generate conditionally:
    - medications.csv if medication rows exist.
    - procedures.csv if procedure rows exist.
    - diagnostic_reports.csv if report rows exist.
    - oncology_timeline.md if there are at least 3 dated oncology-related events (very simple key word matching, so fine just for protoyping. Why 3? With only 1 event, there is no “timeline” — just a single date; with 2, you only have a start and one follow‑up, which is not longitudinal.)

```bash
uv run clinical_synopsis/scripts/generate_derived_documents.py 
```
Output:
```
Processed 50 patients into data/derived/sample50
```


## Build retrieval chunks and metadata DB

`build_retrieval_metadata_db.py` links the human-readable derived documents (in `data/derived/sample50`) and the retrieval system. Instead of just keeping raw text, it creates both metadata records and retrieval chunks (with provenance).

Thus, `build_retrieval_metadata_db.py` is the first part of the ingestion step, the next script `build_minsearch_index.py` does the indexing over that ingested data.


`build_retrieval_metadata_db.py` reads the .md and .csv mock health records for the 50 patients and splits them into chunks, taging each chunk with metadata (patient, doc type, oncology flag, dates, token estimate, FHIR source info) and creating a SQLite database, `data/retrieval/metadata.db`.

```bash
uv run clinical_synopsis/build_retrieval_metadata_db.py
```
Output:
```
{
  "patients": 50,
  "documents": 450,
  "chunks": 117021,
  "db_path": "data/retrieval/metadata.db"
}
```

`metadata.db` has four tables (`patients`, `documents`, `chunks`, `chunk_sources`).  

`chunk_sources` is the table that records, for each chunk of text, which original FHIR resource it came from (resource ID, resource type, source file), so that we can trace every chunk in the mock health records back to original FHIR sources (i.e, the Synthea/mCODE JSON bundles, not the derived CSV/Markdown documents). 

Thus, `file_path` is the path to the derived mock health record (.md/.csv), `title` is the mock label (the filename), `source_file` in `documents` is a placeholder (currently NULL, but could be the name of the bundle if adding extra oncology results from a different dataset, such as genetics), whereas `source_file` in `chunk_sources` is populated where CSV rows carry FHIR provenance.

In [14]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")

conn = sqlite3.connect(db_path)

# List tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)
print(tables)

# Preview patients
patients_df = pd.read_sql_query(
    "SELECT * FROM patients LIMIT 10;",
    conn
)
display(patients_df)

# Preview documents
documents_df = pd.read_sql_query(
    "SELECT * FROM documents LIMIT 10;",
    conn
)
display(documents_df)

# Preview chunks for one patient
example_patient = patients_df.iloc[0]["patient_id"]
chunks_df = pd.read_sql_query(
    "SELECT * FROM chunks WHERE patient_id = ? LIMIT 20;",
    conn,
    params=[example_patient],
)
display(chunks_df)

conn.close()

            name
0  chunk_sources
1         chunks
2      documents
3       patients


,patient_id
0,03b93198-d95e-c385-c3a7-80470f411d18
1,0c0f2095-e8ab-7ac4-6ef4-625748255480
2,0f5704ee-b38b-5a68-449d-9c44806517d0
3,188e1f01-15b7-d51b-c76d-bdd7772a10e9
4,25197dc8-9425-1999-5914-f2171b0d4e32
5,263375ec-5856-81b8-9e51-1cb8e8bcba30
6,29f6beee-162f-0113-7884-72245814693f
7,397b2de6-ccd8-858f-bf4a-b6fc379589bd
8,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678
9,3af995f1-02a5-07ee-5a7e-e2470a017f1e


,document_id,patient_id,doc_type,file_path,source_file,title,format,created_at,is_oncology,date_start,date_end
0,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,conditions,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,conditions.csv,csv,2026-07-30T09:30:20.264942+00:00,0,NaN,NaN
1,924497c65c7653ab43fcac9b6bad5ed5269da152,03b93198-d95e-c385-c3a7-80470f411d18,diagnostic_reports,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,diagnostic_reports.csv,csv,2026-07-30T09:30:20.266050+00:00,0,NaN,NaN
2,35272104bf8dd56da951d0f61a2566a95137a8cb,03b93198-d95e-c385-c3a7-80470f411d18,encounters,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,encounters.csv,csv,2026-07-30T09:30:20.269138+00:00,0,NaN,NaN
3,0d3f88ac69f2b8dab7233162ca3e1d7d86a51afb,03b93198-d95e-c385-c3a7-80470f411d18,medications,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,medications.csv,csv,2026-07-30T09:30:20.272446+00:00,0,NaN,NaN
4,aed15b6fe0ca28c5b8b8afae27e02c598901b64c,03b93198-d95e-c385-c3a7-80470f411d18,observations,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,observations.csv,csv,2026-07-30T09:30:20.273049+00:00,0,NaN,NaN
5,eacfd84f2024e811caae390e056a4e52fefc5249,03b93198-d95e-c385-c3a7-80470f411d18,oncology_timeline,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,Oncology Timeline: Aurora248 Dooley940,md,2026-07-30T09:30:20.280433+00:00,1,2009-10-21T08:03:53-04:00,2018-06-27T07:44:44-04:00
6,304ddea3daed73b3c0c0f47535fef1906ea22987,03b93198-d95e-c385-c3a7-80470f411d18,oncology_timeline_events,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,oncology_timeline_events.csv,csv,2026-07-30T09:30:20.280800+00:00,1,NaN,NaN
7,c869d9ad3b6b47f02a56ebcdf021c6140f287c51,03b93198-d95e-c385-c3a7-80470f411d18,patient_overview,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,Patient Overview: Aurora248 Dooley940,md,2026-07-30T09:30:20.281644+00:00,0,1997-10-24,2022-06-10T08:18:53-04:00
8,2aad94f00f803b0892c82120591c681d286d21c6,03b93198-d95e-c385-c3a7-80470f411d18,procedures,data/derived/sample50/03b93198-d95e-c385-c3a7-...,None,procedures.csv,csv,2026-07-30T09:30:20.281921+00:00,0,NaN,NaN
9,df09e41967a4bd2df8b60b265543295a83eb02d4,0c0f2095-e8ab-7ac4-6ef4-625748255480,conditions,data/derived/sample50/0c0f2095-e8ab-7ac4-6ef4-...,None,conditions.csv,csv,2026-07-30T09:30:20.286011+00:00,0,NaN,NaN


,chunk_id,document_id,patient_id,chunk_index,chunk_text,char_start,char_end,token_estimate,heading,is_oncology,date_start,date_end
0,a6cbd84e70b862c31d452c69c1ef9d8ea9088385,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,0,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,570,142,conditions,0,1998-08-22T08:03:53-04:00,1998-10-23T09:44:53-04:00
1,3c8d89b9d85c63310a8a2a091adf081e708ecd14,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,1,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,554,138,conditions,0,1999-05-17T08:03:53-04:00,1999-10-01T08:03:53-04:00
2,1dc25ebe6124ce3cff281500849d54c438ec8ad2,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,2,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,570,142,conditions,0,1999-10-10T09:44:53-04:00,1999-11-25T10:38:53-05:00
3,c71f43a6030ad74e4c10e17be37fc0efa212f490,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,3,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,554,138,conditions,0,2000-05-28T08:03:53-04:00,2000-09-29T08:03:53-04:00
4,7cbc65f314a5c964fa77e6c989e3bfb7206c118b,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,4,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,583,145,conditions,0,2000-06-18T08:03:53-04:00,2000-07-04T08:03:53-04:00
5,fc0f920bd8c5e5539f8a8f50500ddaa595c0df49,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,5,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,602,150,conditions,0,2002-02-04T03:03:53-05:00,2002-02-15T14:03:53-05:00
6,b1e42675da04a91a64c1f87f3473b0ddc3c59c6a,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,6,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,583,145,conditions,0,2003-03-27T03:03:53-05:00,2003-04-08T04:03:53-04:00
7,fbfa752dcca09b15589dc06f1e6bc6389dd66102,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,7,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,554,138,conditions,0,2003-07-16T08:03:53-04:00,2003-10-10T08:03:53-04:00
8,f351687b0d8d395db145b50fadbce3aefb5564df,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,8,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,614,153,conditions,1,2009-10-21T08:03:53-04:00,2009-10-21T09:03:53-04:00
9,493bd34365179e6a9ca8fdcadd878618977d7032,cf3c3f6b90feff89b9788eabf6fc8c9f7f8c07ee,03b93198-d95e-c385-c3a7-80470f411d18,9,patient_id: 03b93198-d95e-c385-c3a7-80470f411d...,0,591,147,conditions,0,2009-10-21T08:33:53-04:00,2009-10-21T09:03:53-04:00


## Build minsearch index and vector index

This the second step of ingestion.  

To reproduce the index from scratch, run `uv run scripts/build_retrieval_metadata_db.py` and `uv run scripts/build_minsearch_index.py`, then `uv run scripts/build_vector_index.py`.

For review, you can skip this and go straight to `rag.py` since precomputed indexes are included in `data/retrieval/minsearch_index.pkl`, `data/retrieval/vector_index.npz` and `data/retrieval/vector_index_metadata.json`.



`build_minsearch_index.py` reads all chunk metadata from `metadata.db`, turns each chunk into a document dictionary, and then builds a lexical search index over these documents (`text_fields=["title", "heading", "chunk_text"]`, `keyword_fields=["patient_id", "doc_type", "is_oncology"]`), and then saves the index to a pickle file `data/retrieval/minsearch_index.pkl`. 

The script also writes the document list to `data/retrieval/minsearch_documents.json`, which can be used for rebuilding the lexical index without going back to the SQLite database.


```bash
uv run clinical_synopsis/build_minsearch_index.py
```
Output:
```
Built minsearch index for 117021 chunks
Saved index to data/retrieval/minsearch_index.pkl
Saved documents to data/retrieval/minsearch_documents.json
```


`build_vector_index.py` creates ONNX-based embeddings for all chunks, using [`embedder.py`](https://github.com/DataTalksClub/llm-zoomcamp/blob/c85fa440791efd6c902f7f30ccedaae4483e3f9e/02-vector-search/embed/embedder.py) from the course.  It loads the chunk data from `metadata.db`, encodes each chunk’s text into a vector using Embedder, and saves the embedding matrix and aligned chunk IDs to `vector_index.npz`. It also stores the corresponding chunk metadata plus embedding settings in `vector_index_metadata.json`, so `rag.py` can later do semantic and hybrid search over those chunks.

Batch processing: each text (one per chunk) is sliced into a batch of 64 texts, which are encoded by the embedder as a group. The default batch size is 64, change it to 16 or 32 to improve CPU/RAM performance.


## Run RAG

`rag.py` contains the retrieval and RAG logic (the job done by [`rag_helper.py`](https://github.com/DataTalksClub/llm-zoomcamp/blob/c85fa440791efd6c902f7f30ccedaae4483e3f9e/01-agentic-rag/code/rag_helper.py#L4)  in the course). Similarly to [`rag.py`](https://github.com/alexeygrigorev/fitness-assistant/blob/main/fitness_assistant/rag.py) in the sample capstone project, it also includes evaluation, timing, and cost calculation.

It loads both a lexical minsearch index and a semantic embedding index, retrieves the most relevant patient chunks using lexical, semantic, or hybrid search, builds a context prompt for the LLM, generates an answer grounded in that context, and then runs an LLM-based evaluation of the answer’s relevance and groundedness while also tracking token usage, latency, and cost.

For lexical search `rag.py` loads `minsearch_index.pkl`, for semantic_search and hybrid_search `rag.py` loads the `vector_index.npz` and `vector_index_metadata.json` files (in `data/retrieval`).

The function `evaluate_relevance()` uses the LLM as a judge to classify each generated answer for: 
- relevance to the question,
- groundedness in the retrieved context.


answer relevance,
groundedness/faithfulness,
retrieval quality,
context usefulness



# Note on COMPLEXITY SCORES for each patient

A higher complexity score should reflect more encounters, conditions, procedures, meds, reports for a given patient, as well as a longer follow-up period (e.g., Febrile neutropenia condition gives a clear date (onsetDateTime, recordedDate) and is tied to an encounter, which contributes to follow-up and complexity).

In `sample_mcode_patients.py` a complexity score is computed as:
```python
complexity_score = (
    counts["Encounter"] * 3
    + counts["Observation"] * 1
    + counts["Condition"] * 2
    + counts["Procedure"] * 2
    + counts["MedicationRequest"] * 2
    + counts["MedicationAdministration"] * 2
    + counts["DiagnosticReport"] * 2
    + min(followup_days // 180, 20)
)
```
Which means that:
- Each resource type contributes with a **weight**:
  - Encounters: \(3 \times\) number of encounters (heavier weight).
  - Observations: \(1 \times\) number of observations.
  - Conditions, Procedures, MedicationRequest, MedicationAdministration, DiagnosticReport: each \(2 \times\) their counts.
- Plus a **time component**:
  - `followup_days` is the difference between the first and last clinical dates found in the bundle.
  - `followup_days // 180` converts follow-up into “half-year blocks”.
  - This term is capped at 20, so very long records don’t dominate.

